In [1]:
import numpy as np
import networkx as nx
from copy import deepcopy
from numpy import linalg as LA
from scipy.stats import wasserstein_distance_nd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import utils as ut

# autoreload for development
%load_ext autoreload
%autoreload 2

construct 2 2D distributions of transition probabilities in a graph with 10 nodes, such that:
1. the 2nd largest eigenvalue of both distributions is equal
2. the wasserstein distance between each distribution and the uniform distribution is very different

In [12]:
def get_second_largest_laplacian_eigenvalue(M):
    # D = np.diag(np.sum(M, axis=1))
    I = np.eye(M.shape[0])
    L = I - M
    eigenvalues = LA.eigvals(L)  # only eigenvalues; avoids mixed tuple
    eigenvalues = np.real(eigenvalues)
    eigenvalues = np.sort(eigenvalues)
    # use Fiedler value when available; fallback to first eigenvalue for 1x1
    lambda_2 = eigenvalues[1] if eigenvalues.size > 1 else eigenvalues[0]
    return lambda_2

def get_wasserstein_distance_from_uniform(M):
    uniform_dist = np.ones(M.shape) / M.size
    d = wasserstein_distance_nd(M, uniform_dist)
    return d

In [30]:
def plot_distributions(distribution_list, show=False):
    fig = make_subplots(rows=2, cols=len(distribution_list))

    for di, distribution in enumerate(distribution_list):

        # plot (1, i) - heatmap
        fig.add_trace(go.Heatmap(z=distribution, colorscale='Viridis', showscale=False), row=1, col=di+1)
        A_wasserstein = get_wasserstein_distance_from_uniform(distribution)
        fig.update_xaxes(title_text=f"W=({A_wasserstein:.4f})", row=1, col=di+1)

        # plot (2, i) - Graph
        nodes = np.arange(distribution.shape[0])
        edges = []
        for i in range(distribution.shape[0]):
            for j in range(distribution.shape[1]):
                if distribution[i, j] > 0:
                    edges.append((nodes[i], nodes[j], distribution[i, j]))
        G = nx.DiGraph()
        G.add_nodes_from(nodes)
        G.add_weighted_edges_from(edges)
        fig_g = ut.draw_graph(G)
        for tr in fig_g.data:
            fig.add_trace(deepcopy(tr), row=2, col=di+1)
        A_lambda2 = get_second_largest_laplacian_eigenvalue(distribution)
        fig.update_xaxes(title_text=f"λ2=({A_lambda2:.4f})", row=2, col=di+1)

    fig.update_layout(height=800, width=400 * len(distribution_list), showlegend=False)

    if show:
        fig.show(renderer='browser')
        
    return fig


In [13]:
A = np.ones((10, 10)) * 0.05 + np.eye(10) * 0.5
B = np.vstack([np.ones(10) * 0.5, np.zeros((9, 10))]).reshape(10, 10).T + np.eye(10) * 0.5

plot_distributions(A, B)

In [31]:
A = np.array([[1, 3, 2, 1, 1, 1, 1, 1, 1, 1],
              [3, 1, 3, 2, 1, 1, 1, 1, 1, 1],
              [2, 3, 1, 3, 2, 1, 1, 1, 1, 1],
              [1, 2, 3, 1, 3, 2, 1, 1, 1, 1],
              [1, 1, 2, 3, 1, 3, 2, 1, 1, 1],
              [1, 1, 1, 2, 3, 1, 3, 2, 1, 1],
              [1, 1, 1, 1, 2, 3, 1, 3, 2, 1],
              [1, 1, 1, 1, 1, 2, 3, 1, 3, 2],
              [1, 1, 1, 1, 1, 1, 2, 3, 1, 3],
              [1, 1, 1, 1, 1, 1, 1, 2, 3, 1],
              ])
A = (A + np.eye(10) * 5) / np.sum(A, axis=1, keepdims=True)

B = np.array([[1, 3, 2, 1, 1, 1, 1, 1, 1, 1],
              [2, 3, 1, 3, 2, 1, 1, 1, 1, 1],
              [1, 2, 3, 1, 3, 2, 1, 1, 1, 1],
              [1, 1, 2, 3, 1, 3, 2, 1, 1, 1],
              [1, 1, 1, 2, 3, 1, 3, 2, 1, 1],
              [1, 1, 1, 1, 2, 3, 1, 3, 2, 1],
              [1, 1, 1, 1, 1, 2, 3, 1, 3, 2],
              [3, 1, 3, 2, 1, 1, 1, 1, 1, 1],
              [1, 1, 1, 1, 1, 1, 2, 3, 1, 3],
              [1, 1, 1, 1, 1, 1, 1, 2, 3, 1],
              ])
B = (B + np.eye(10) * 5) / np.sum(B, axis=1, keepdims=True)

U = np.ones((10, 10)) / 10

W = np.random.rand(10, 10) / np.sum(B, axis=1, keepdims=True)

plot_distributions([A, U, W])

In [52]:
import os
import librosa

# DPV data
src_folder = '../data/croatia/2407_1_600m'
file_list = os.listdir(src_folder)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

dpv_data = []
for file in file_list[6:-15]:
    _data, fs = librosa.load(os.path.join(src_folder, file))
    dpv_data.append(_data)
# dpv_data = np.concatenate(dpv_data)


# # BG Noise WAVs
# noise_src_folder = '../data/croatia/2507_1_1000m'
# file_list = os.listdir(noise_src_folder)
# file_list = [f for f in file_list if f.endswith('.wav')]
# file_list.sort()

# bg_data = []
# for file in file_list[:5]:
#     _data, fs = librosa.load(os.path.join(noise_src_folder, file))
#     bg_data.append(_data)
# # bg_data = np.concatenate(bg_data)

dpv_data_slice = dpv_data[len(dpv_data)//2]
# bg_data_slice = bg_data[len(bg_data)//2]

dpv_data_slice, fs = librosa.load('../data/ds2/pos/dpv2_1m.wav')

In [53]:
nperseg = 16000
overlap = 0.5
window = 'hann'
dc = 20
crop_freq = None

F, T, Sxx, phasogram = ut.calc_spectrogram(dpv_data_slice, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)
pxx = ut.calc_welch_from_spectrogram(Sxx, normalization_window_size=9)
K_wasserstein = ut.get_all_Ks(phasogram, F, 10, mode='wasserstein')
K_laplacian = ut.get_all_Ks(phasogram, F, 10, mode='laplacian')
fig = make_subplots(rows=1, cols=4, column_widths=[0.7, 0.1, 0.1, 0.1], shared_yaxes=True, column_titles=['Spectrogram', 'Welch', 'Wasserstein', 'Laplacian'])
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showscale=False, showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pxx, y=F, mode='lines', line_color='blue', showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=K_wasserstein, y=F, mode='lines', line_color='red', showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=K_laplacian, y=F, mode='lines', line_color='green', showlegend=False), row=1, col=4)
fig.show()


In [47]:
nperseg = 16000
overlap = 0.5
window = 'hann'
dc = 20
crop_freq = None

F, T, Sxx, phasogram = ut.calc_spectrogram(dpv_data_slice, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)
fix = np.where(F >= 857)[0][0]
nix = np.where(F >= 4000)[0][0]
dpv = phasogram[fix, :]
bg = phasogram[nix, :]
noisy_dpv = dpv + 10 * bg

dpv_dist = ut.get_s2g(dpv, 10)
dpv_dist = dpv_dist / np.sum(dpv_dist, axis=1, keepdims=True)
bg_dist = ut.get_s2g(bg, 10)
bg_dist = bg_dist / np.sum(bg_dist, axis=1, keepdims=True)
noisy_dpv_dist = ut.get_s2g(noisy_dpv, 10)
noisy_dpv_dist = noisy_dpv_dist / np.sum(noisy_dpv_dist, axis=1, keepdims=True)

plot_distributions([dpv_dist, noisy_dpv_dist, bg_dist])